*****************************************************************************
# BIOMASS NC REGION - PART 1
*****************************************************************************

# *Download RAP Data*

Date: 6 February 2025

This Jupyter notebook is part of the Ecosystems Transformation project of Earth Lab at the University of Colorado Boulder. In this script the data from the Rangeland Analysis platform is downloaded from Google Earth Engine using the geemap package.

*Author (of modified script): Esmee Mulder*

## Credits RAP dataset

Conversion of partitioned NPP (annuals forbs and grasses, perennial forbs and grasses) to aboveground biomass as described in Jones et al. (2021).
* Inputs: partitioned NPP, mean annual temperature
* Authors: Nathaniel Robinson, Matthew Jones, Brady Allred
* Contact: Sarah Mccord (sarah.mccord@usda.gov)

Jones, M.O., N.P. Robinson, D.E. Naugle, J.D. Maestas, M.C. Reeves, R.W.
Lankston, and B.W. Allred. 2021. Annual and 16-Day Rangeland Production
Estimates for the Western United States. Rangeland Ecology & Management
77:112–117.

## RAP Datasets

All datasets as shown in Google Earth Engine:
1. projects/rap-data-365417/assets/npp-partitioned-v3
2. projects/rap-data-365417/assets/npp-partitioned-16day-v3
3. projects/rap-data-365417/assets/npp-partitioned-16day-v3-provisional
4. projects/rap-data-365417/assets/vegetation-cover-v3
5. projects/rap-data-365417/assets/gridmet-MAT (mean anual temp)

Annual GeoTIFFs can be downloaded at:
1. http://rangeland.ntsg.umt.edu/data/rap/rap-vegetation-biomass/
2. http://rangeland.ntsg.umt.edu/data/rap/rap-vegetation-cover/
3. http://rangeland.ntsg.umt.edu/data/rap/rap-vegetation-npp/

# 1. Import packages

In [1]:
import ee 
import geemap
import os

# 2. Convert pNPP to AGB
This section on conversion of partitioned Net Primary Production (pNPP) to Above Ground Biomass (AGB) is by Nathaniel Robinson, Matthew Jones, Brady Allred as mentioned in the credits

In [2]:
ee.Authenticate()
ee.Initialize(project='XXX') #insert own google earth engine credentials here

In [3]:
npp = ee.ImageCollection("projects/rap-data-365417/assets/npp-partitioned-v3") \
.select(['afgNPP', 'pfgNPP'])
mat = ee.ImageCollection("projects/rap-data-365417/assets/gridmet-MAT")

Biomass conversion function
* input: two band image (afgNPP, pfgNPP) from projects/rangeland-analysis-platform/npp-partitioned-v2
* output: three band image, aboveground biomass (afgAGB, pfgAGB, herbaceousAGB)

In [1]:
def func_fww(image):

    year = ee.Date(image.get('system:time_start')).format('YYYY')
    matYear = mat.filterDate(year).first()
    fANPP = (matYear.multiply(0.0129)).add(0.171).rename('fANPP'); 

    #choose factor for multiplication based on desired output units:
    
    #multiply(0.0001) \ #NPP scalar
    #.multiply(2.20462) \ #KgC to LbsC
    #.multiply(0.001) \ #from Kg to Mg
    #.multiply(4046.86) \ #m2 to acres
    #.multiply(10000) \ #m2 to ha
    #.multiply(fANPP) \ #fraction of NPP aboveground
    #.multiply(2.1276) \ #C to biomass
    
    agb = image.multiply(0.0001) \
    .multiply(0.001) \
    .multiply(10000) \
    .multiply(fANPP) \
    .multiply(2.1276) \
    .rename(['afgAGB', 'pfgAGB']) \
    .copyProperties(image, ['system:time_start']) \
    .set('year', year)

    herbaceous = ee.Image(agb).reduce(ee.Reducer.sum()).rename(['herbaceousAGB'])

    agb = ee.Image(agb).addBands(herbaceous)

    return agb

In [5]:
biomassFunction = func_fww

In [6]:
biomass = npp.map(biomassFunction)

# 3. Visualize biomass data (using geemap)

In [7]:
# add 2019 perennial forb and grass biomass (pfgAGB) to map
# bamako palette, from 'users/gena/packages:palettes'
bamakoReverse = ["FFE599", "E3C961", "B9A525", "878E03", "617E14", "436A25", "2B5A34", "154C41","00404D"]    

In [8]:
m = geemap.Map(center=(39.7631584037253, -101.59812927246095), zoom=4)
m

Map(center=[39.7631584037253, -101.59812927246095], controls=(WidgetControl(options=['position', 'transparent_…

In [9]:
m.addLayer(biomass.filterDate('2019').select('pfgAGB'),
{'min': 0, 'max': 5, 'palette': bamakoReverse},
'perennials2019')

In [10]:
vis_params = {
    "min": 0,
    "max": 5,
    "palette": bamakoReverse,
}
colors = vis_params["palette"]
vmin = vis_params["min"]
vmax = vis_params["max"]
label = 'Biomass Mg/ha'

m.add_colorbar(vis_params)

In [11]:
# Overview of the different data sets
npp_year = ee.ImageCollection("projects/rap-data-365417/assets/npp-partitioned-v3")
npp_16 = ee.ImageCollection("projects/rap-data-365417/assets/npp-partitioned-16day-v3")
npp_16_prov = ee.ImageCollection("projects/rap-data-365417/assets/npp-partitioned-16day-v3-provisional")
veg_cover = ee.ImageCollection('projects/rap-data-365417/assets/vegetation-cover-v3')

# 4. Clip the biomass data to the NC region

In [12]:
NC_shp = ".../NC_CASC_region.shp" #insert path to NC region shapefile
NC = geemap.shp_to_ee(NC_shp)
m.addLayer(NC, {}, "North Central Region")

In [13]:
# clip to shapefile
def clip_to_shapefile_NC(img):
    return img.clip(NC.geometry())

biomass_clip = (
   biomass
    .filterBounds(NC) # filter bounds using roi
    .map(clip_to_shapefile_NC)
)

m.addLayer(biomass_clip.filterDate('2018').select('pfgAGB'),
{'min': 0, 'max': 5, 'palette': bamakoReverse},
'perennials2018')

In [14]:
#check output
biomass_clip.first().bandNames().getInfo()

['afgAGB', 'pfgAGB', 'herbaceousAGB']

# 5. Download the raster files

In [25]:
#The full biomass data over the NC region is too large to download
#create a fishnet to download in paralel
fishnet = geemap.fishnet(NC, h_interval=1.5, v_interval=1, delta=0.9)
style = {"color": "ffff00ff", "fillColor": "00000000"}
m.addLayer(fishnet.style(**style), {}, "Fishnet")

In [18]:
# Define the start and end years
start_year = 2003
end_year = 2017

# Loop through the years and process the images 
for year in range(start_year, end_year):
    start_date = str(year)
    end_date = str(year + 1)
    out_dir = f"BM_tiles_100_Mg_ha_{start_date}"
    mosaic_file = f"Rangeland_100_BM_Mg_ha_{start_date}.tif"
    
    biomass_download = biomass_clip.filterDate(start_date, end_date)
    image = biomass_download.first()
    geemap.download_ee_image_tiles_parallel(image, fishnet, out_dir=out_dir, scale=100)
    geemap.mosaic(out_dir, mosaic_file)

Finished in 551.378515958786 seconds.
Reading 1/38: 01.tif
Reading 2/38: 02.tif
Reading 3/38: 03.tif
Reading 4/38: 04.tif
Reading 5/38: 05.tif
Reading 6/38: 06.tif
Reading 7/38: 07.tif
Reading 8/38: 08.tif
Reading 9/38: 09.tif
Reading 10/38: 10.tif
Reading 11/38: 11.tif
Reading 12/38: 12.tif
Reading 13/38: 13.tif
Reading 14/38: 14.tif
Reading 15/38: 15.tif
Reading 16/38: 16.tif
Reading 17/38: 17.tif
Reading 18/38: 18.tif
Reading 19/38: 19.tif
Reading 20/38: 20.tif
Reading 21/38: 21.tif
Reading 22/38: 22.tif
Reading 23/38: 23.tif
Reading 24/38: 24.tif
Reading 25/38: 25.tif
Reading 26/38: 26.tif
Reading 27/38: 27.tif
Reading 28/38: 28.tif
Reading 29/38: 29.tif
Reading 30/38: 30.tif
Reading 31/38: 31.tif
Reading 32/38: 32.tif
Reading 33/38: 33.tif
Reading 34/38: 34.tif
Reading 35/38: 35.tif
Reading 36/38: 36.tif
Reading 37/38: 37.tif
Reading 38/38: 38.tif
Merging rasters...
Finished in 311.38867688179016 seconds.
Reading 1/38: 01.tif
Reading 2/38: 02.tif
Reading 3/38: 03.tif
Reading 4/38:

In [20]:
#paste the tiles together in one file
biomass_download = biomass_reproj.filterDate("2000", "2001")
image = biomass_download.first()
geemap.download_ee_image_tiles_parallel(
    image, fishnet, out_dir="tiles_2000", scale=500)
geemap.mosaic("tiles_2000", "2000_mosaic.tif")
biomass_download = biomass_reproj.filterDate("2001", "2002")
image = biomass_download.first()
geemap.download_ee_image_tiles_parallel(
    image, fishnet, out_dir="tiles_2001", scale=500)
geemap.mosaic("tiles_2001", "2001_mosaic.tif")
biomass_download = biomass_reproj.filterDate("2002", "2003")
image = biomass_download.first()
geemap.download_ee_image_tiles_parallel(
    image, fishnet, out_dir="tiles_2002", scale=500)
geemap.mosaic("tiles_2002", "2002_mosaic.tif")
biomass_download = biomass_reproj.filterDate("2003", "2004")
image = biomass_download.first()
geemap.download_ee_image_tiles_parallel(
    image, fishnet, out_dir="tiles_2003", scale=500)
geemap.mosaic("tiles_2003", "2003_mosaic.tif")
biomass_download = biomass_reproj.filterDate("2004", "2005")
image = biomass_download.first()
geemap.download_ee_image_tiles_parallel(
    image, fishnet, out_dir="tiles_2004", scale=500)
geemap.mosaic("tiles_2004", "2004_mosaic.tif")
biomass_download = biomass_reproj.filterDate("2005", "2006")
image = biomass_download.first()
geemap.download_ee_image_tiles_parallel(
    image, fishnet, out_dir="tiles_2005", scale=500)
geemap.mosaic("tiles_2005", "2005_mosaic.tif")
biomass_download = biomass_reproj.filterDate("2006", "2007")
image = biomass_download.first()
geemap.download_ee_image_tiles_parallel(
    image, fishnet, out_dir="tiles_2006", scale=500)
geemap.mosaic("tiles_2006", "2006_mosaic.tif")
biomass_download = biomass_reproj.filterDate("2007", "2008")
image = biomass_download.first()
geemap.download_ee_image_tiles_parallel(
    image, fishnet, out_dir="tiles_2007", scale=500)
geemap.mosaic("tiles_2007", "2007_mosaic.tif")
biomass_download = biomass_reproj.filterDate("2008", "2009")
image = biomass_download.first()
geemap.download_ee_image_tiles_parallel(
    image, fishnet, out_dir="tiles_2008", scale=500)
geemap.mosaic("tiles_2008", "2008_mosaic.tif")
biomass_download = biomass_reproj.filterDate("2009", "2010")
image = biomass_download.first()
geemap.download_ee_image_tiles_parallel(
    image, fishnet, out_dir="tiles_2009", scale=500)
geemap.mosaic("tiles_2009", "2009_mosaic.tif")
biomass_download = biomass_reproj.filterDate("2010", "2011")
image = biomass_download.first()
geemap.download_ee_image_tiles_parallel(
    image, fishnet, out_dir="tiles_2010", scale=500)
geemap.mosaic("tiles_2010", "2010_mosaic.tif")
biomass_download = biomass_reproj.filterDate("2011", "2012")
image = biomass_download.first()
geemap.download_ee_image_tiles_parallel(
    image, fishnet, out_dir="tiles_2011", scale=500)
geemap.mosaic("tiles_2011", "2011_mosaic.tif")
biomass_download = biomass_reproj.filterDate("2012", "2013")
image = biomass_download.first()
geemap.download_ee_image_tiles_parallel(
    image, fishnet, out_dir="tiles_2012", scale=500)
geemap.mosaic("tiles_2012", "2012_mosaic.tif")
biomass_download = biomass_reproj.filterDate("2013", "2014")
image = biomass_download.first()
geemap.download_ee_image_tiles_parallel(
    image, fishnet, out_dir="tiles_2013", scale=500)
geemap.mosaic("tiles_2013", "2013_mosaic.tif")
biomass_download = biomass_reproj.filterDate("2014", "2015")
image = biomass_download.first()
geemap.download_ee_image_tiles_parallel(
    image, fishnet, out_dir="tiles_2014", scale=500)
geemap.mosaic("tiles_2014", "2014_mosaic.tif")
biomass_download = biomass_reproj.filterDate("2015", "2016")
image = biomass_download.first()
geemap.download_ee_image_tiles_parallel(
    image, fishnet, out_dir="tiles_2015", scale=500)
geemap.mosaic("tiles_2015", "2015_mosaic.tif")
biomass_download = biomass_reproj.filterDate("2016", "2017")
image = biomass_download.first()
geemap.download_ee_image_tiles_parallel(
    image, fishnet, out_dir="tiles_2016", scale=500)
geemap.mosaic("tiles_2016", "2016_mosaic.tif")
biomass_download = biomass_reproj.filterDate("2017", "2018")
image = biomass_download.first()
geemap.download_ee_image_tiles_parallel(
    image, fishnet, out_dir="tiles_2017", scale=500)
geemap.mosaic("tiles_2017", "2017_mosaic.tif")

NameError: name 'biomass_reproj' is not defined